# Practice - 04 - COPY INTO and file-level idempotency

Prerequisites / execution context: Run 00. Creates a fresh isolated input path and target table each time the configuration below runs.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R09: COPY INTO tutorial](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/copy-into); [R08: Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes/).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Load a JSON file incrementally

Create isolated variables run_id, source and T under the lab volume/Bronze schema. Write part1.json with (id=1,amount=10),(id=2,amount=20). Create the Delta target with BIGINT columns. Build cmd as the COPY INTO SQL, run it and count rows.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
import uuid
run_id = uuid.uuid4().hex[:10]
source = f"{VOLUME}/copy_{run_id}"
T = f"{BRONZE}.copy_{run_id}"
dbutils.fs.put(f"{source}/part1.json", '{"id":1,"amount":10}\n{"id":2,"amount":20}', True)

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(T).count() == 2
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIENyZWF0ZSB0d28gaW1tdXRhYmxlIHNvdXJjZSBmaWxlcwpBIG5ldyBleHBlcmltZW50IElEIGF2b2lkcyBjb21iaW5pbmcgYW4gb2xkIHRhcmdldCBoaXN0b3J5IHdpdGggYSBmcmVzaCBpbnB1dCBsb2NhdGlvbi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmltcG9ydCB1dWlkCnJ1bl9pZD11dWlkLnV1aWQ0KCkuaGV4WzoxMF0Kc291cmNlPWYmcXVvdDt7Vk9MVU1FfS9jb3B5X3tydW5faWR9JnF1b3Q7ClQ9ZiZxdW90O3tCUk9OWkV9LmNvcHlfe3J1bl9pZH0mcXVvdDsKZGJ1dGlscy5mcy5wdXQoZiZxdW90O3tzb3VyY2V9L3BhcnQxLmpzb24mcXVvdDssJiN4Mjc7eyZxdW90O2lkJnF1b3Q7OjEsJnF1b3Q7YW1vdW50JnF1b3Q7OjEwfVxueyZxdW90O2lkJnF1b3Q7OjIsJnF1b3Q7YW1vdW50JnF1b3Q7OjIwfSYjeDI3OyxUcnVlKQpzcGFyay5zcWwoZiZxdW90O0NSRUFURSBUQUJMRSB7VH0gKGlkIEJJR0lOVCwgYW1vdW50IEJJR0lOVCkgVVNJTkcgREVMVEEmcXVvdDspCmNtZD1mJnF1b3Q7Q09QWSBJTlRPIHtUfSBGUk9NICYjeDI3O3tzb3VyY2V9JiN4Mjc7IEZJTEVGT1JNQVQgPSBKU09OJnF1b3Q7CnNwYXJrLnNxbChjbWQpLnNob3coKQphc3NlcnQgc3BhcmsudGFibGUoVCkuY291bnQoKT09Mgo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 2: Prove file tracking is not business deduplication

Rerun cmd on unchanged input. Then create part2.json with (id=2,amount=20),(id=3,amount=30), run cmd again, and compare total rows with distinct IDs. Explain both outcomes.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(T).count() == 4
assert spark.table(T).select("id").distinct().count() == 3
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIFJlLXJ1biB3aXRob3V0IGEgbmV3IGZpbGUKRXhwZWN0IHRoZSBzYW1lIHR3byByb3dzLiBUaGUgY29tbWFuZCB0cmFja3MgbG9hZGVkIGZpbGVzOyBpdCBkb2VzIG5vdCBkZWR1cGxpY2F0ZSBidXNpbmVzcyBrZXlzIGFjcm9zcyBkaWZmZXJlbnQgZmlsZXMuCgoKVGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpcyBzaG93biBuZXh0LiBDb21wYXJlIHlvdXIgZGVzaWduIGFuZCByZXN1bHRzIGJlZm9yZSBjb3B5aW5nIGl0LjwvZGl2PjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPnNwYXJrLnNxbChjbWQpLnNob3coKQphc3NlcnQgc3BhcmsudGFibGUoVCkuY291bnQoKT09MgpkYnV0aWxzLmZzLnB1dChmJnF1b3Q7e3NvdXJjZX0vcGFydDIuanNvbiZxdW90OywmI3gyNzt7JnF1b3Q7aWQmcXVvdDs6MiwmcXVvdDthbW91bnQmcXVvdDs6MjB9XG57JnF1b3Q7aWQmcXVvdDs6MywmcXVvdDthbW91bnQmcXVvdDs6MzB9JiN4Mjc7LFRydWUpCnNwYXJrLnNxbChjbWQpLnNob3coKQphc3NlcnQgc3BhcmsudGFibGUoVCkuY291bnQoKT09NAphc3NlcnQgc3BhcmsudGFibGUoVCkuc2VsZWN0KCZxdW90O2lkJnF1b3Q7KS5kaXN0aW5jdCgpLmNvdW50KCk9PTMKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Written / design task 1

Predict the count of business ID 2 after loading the second file. Explain what COPY INTO tracks and how intentional file reloading affects duplicates.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgQ2hhbGxlbmdlIGFuZCBzb2x1dGlvbgpXaHkgZG9lcyBpZCAyIGFwcGVhciB0d2ljZT8gSXQgY2FtZSBmcm9tIHR3byBkaWZmZXJlbnQgZmlsZXMuIEFwcGx5IGJ1c2luZXNzLWtleSBkZWR1cGxpY2F0aW9uIGRvd25zdHJlYW0uIEZvciBhbiBpbnRlbnRpb25hbCByZWxvYWQsIGludmVzdGlnYXRlIENPUFlfT1BUSU9OUyBmb3JjZSBpbiB0aGUgY29tbWFuZCByZWZlcmVuY2UgYmVmb3JlIHVzaW5nIGl0OyByZXByb2Nlc3NpbmcgY2FuIGR1cGxpY2F0ZSByZWNvcmRzLiBFeHBsYWluIHdoZW4gQXV0byBMb2FkZXIgaXMgYSBiZXR0ZXIgZml0IGZvciBmaWxlIGRpc2NvdmVyeSBhdCBzY2FsZS4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2luZ2VzdGlvbi9jbG91ZC1vYmplY3Qtc3RvcmFnZS9hdXRvLWxvYWRlciIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIxMDogQXV0byBMb2FkZXI8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2luZ2VzdGlvbi9jbG91ZC1vYmplY3Qtc3RvcmFnZS9hdXRvLWxvYWRlci9maWxlLWRldGVjdGlvbi1tb2RlcyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIxMjogQXV0byBMb2FkZXIgZmlsZSBkZXRlY3Rpb24gbW9kZXM8L2E+Lgo8L2Rpdj48L2RldGFpbHM+')
